### Parameter Efficient Fine-Tuning
In this notebook, you're gonna fine-tune large language models within limited GPU memory.

In [ ]:
# Original library versions
# %pip install --quiet transformers==4.34.1 accelerate==0.24.0 sentencepiece==0.1.99 optimum==1.13.2 peft==0.5.0 bitsandbytes==0.41.2.post2

# Preferred versions for Colab as of October 2025 (thanks, Lev!)
# %pip install --quiet "bitsandbytes==0.45.3" "transformers>=4.43,<4.46" "accelerate>=0.33,<0.36" "peft>=0.11.1" "optimum>=1.20.0" "sentencepiece"

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import datasets

import transformers
from tqdm.auto import tqdm, trange
assert torch.cuda.is_available(), "you need cuda for this part"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
model_name = 'Enoch/llama-7b-hf'

# loading Llama tokenizer ...
tokenizer = transformers.LlamaTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message


In [3]:
# ... and the model itself
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto',
    low_cpu_mem_usage=True,
    offload_state_dict=True,
    load_in_4bit=True,
    torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False

model.gradient_checkpointing_enable()  # only store a small subset of activations, re-compute the rest.
model.enable_input_require_grads()     # override an implementation quirk in gradient checkpoints that disables backprop unless inputs require grad
# more on gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

### Prompt tuning: the story of a fox (1 point)

![img](https://i.imgur.com/Ux3qQAu.png) (source: theodd1souts.fandom.com)

In [4]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: <s>A quick brown fox jumps over the lazy dog.
A quick


What a blatant lie! This particular fox assures you that it didn't in fact jump over the lazy dog. No, sir! The fox was just minding its own business. __Your task is to train the model to say truth: no dog was jumped over today.__

In [5]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.0729, device='cuda:0', grad_fn=<NllLossBackward0>)


Except, we can't train the entire model - that would be 28GB gradients in float32. Instead, let's run [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [6]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    To perform prompt tuning, you will need to replace the model's original word embeddings with a layer - THIS layer
    - that inserts trainable prompts instead of the first N token embeddings.
    """

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True
        )

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq_length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), (
            "Don't forget to prepend several BOS tokens to input_ids"
        )

        # Embed the input_ids using the original word embeddings
        input_embeddings = self.original_word_embeddings(input_ids)                     # Shape: [batch_size, seq_length, embedding_dim]

        # Replace the first num_prompts token embeddings with the learnable prompts
        batch_size = input_ids.shape[0]
        learnable_prompts_expanded = self.learnable_prompts.expand(batch_size, -1, -1)  # Shape: [batch_size, num_prompts, embedding_dim]
        remaining_embeddings = input_embeddings[:, self.num_prompts:, :]                # Shape: [batch_size, seq_length - num_prompts, embedding_dim]

        # Concatenate learnable prompts with the embeddings of the remaining tokens
        output_embeddings = torch.cat([learnable_prompts_expanded, remaining_embeddings], dim=1)

        return output_embeddings


In [7]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.cuda.amp.autocast():
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


C:\Users\Vlad\AppData\Local\Temp\ipykernel_1196\1152930240.py:9: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


__Now that it works,__ let's inject learnable prompts into the main model and teach it about foxes.

In [8]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [9]:
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

outputs = model(**batch)
next_word_logits = outputs.logits[:, num_prompts : -1, :]
true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))
print("Loss:", loss)

# raise NotImplementedError("Your task: iteratively train the model to reduce loss using prompt optimizer (opt)")

Loss: tensor(7.6354, device='cuda:0', grad_fn=<NllLossBackward0>)


In [10]:
num_steps = 200
for step in range(1, num_steps+1):
    opt.zero_grad()

    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]

    loss = F.cross_entropy(
        next_word_logits.flatten(0, 1),
        true_next_tokens.flatten(0, 1)
    )

    loss.backward()
    opt.step()

    if step % 20 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")


Step 20: loss = 2.0191
Step 40: loss = 0.1750
Step 60: loss = 0.0300
Step 80: loss = 0.0133
Step 100: loss = 0.0086
Step 120: loss = 0.0064
Step 140: loss = 0.0050
Step 160: loss = 0.0040
Step 180: loss = 0.0033
Step 200: loss = 0.0028


In [11]:
# Final loss assertion
assert loss.item() <= 0.1
print("Good job!")

Good job!


In [12]:
prompt = 'A quick brown fox'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))

# if you did everything right, the model will deny that the fox jumped over the lazy dog


Output: <s>A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Using HuggingFace PEFT (2 point)

[`peft`](https://huggingface.co/docs/peft/index) is a transformer's sister library that allows you to apply various __p__arameter __e__fficient __f__ine-__t__uning methods to pre-trained transformers. The library imlements both prompt tuning, prefix tuning, as well as several adapter-based techniques under a common interface:



In [ ]:
# Здесь у меня не хотела освобождаться память видеокарты, поэтому я перезагрузил kernel:
# Нумерация порядка выполнения ячеек начинается заново, и такое еще было пару раз позже...

In [4]:
import peft
assert isinstance(model.model.embed_tokens, nn.Embedding), "please reload the model"

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 65536
Total parameters (excluding quantization): 3500478464


In [ ]:
# Your task: optimize the PEFT-wrapped model to achieve next token prediction loss < 0.1, but this time using PEFT
# Please note: you no longer need to prepend PAD tokens, but you still need to skip :num_virtual_tokens: first logits.
# Finally, generate the sentence to make sure that the model learned the truth.

In [5]:
# Define the ground truth sentence
# the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
the_truth = "A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it anyway!"
batch = tokenizer(the_truth, return_tensors="pt", return_token_type_ids=False).to(device)

In [6]:
# Training Configuration
loss_threshold = 0.1    # Desired loss threshold
num_epochs = 100        # Max number of epochs
learning_rate = 0.1     # Learning rate - higher for prompt tuning - we're only training 16 tokens

# Define the optimizer for trainable parameters (PEFT prompts)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training Loop
for epoch in range(num_epochs):
    # Forward pass
    outputs = model(**batch)
    
    # Skip logits for virtual tokens and the last token
    next_word_logits = outputs.logits[:, peft_config.num_virtual_tokens:-1, :]  # Skip virtual tokens
    true_next_tokens = batch['input_ids'][:, 1:]                                # Shift ground truth tokens by one
    
    # Compute the loss
    loss = F.cross_entropy(
        next_word_logits.reshape(-1, next_word_logits.size(-1)),    # Flatten logits
        true_next_tokens.reshape(-1)                                # Flatten ground truth tokens
    )
    
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Print loss for tracking
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")
    
    # Stop training if loss is below threshold
    if loss.item() < loss_threshold:
        print("Loss threshold reached. Stopping training.")
        break
else:
    print("Maximum epochs reached without meeting the loss threshold.")

Epoch 1/100, Loss: 7.073969841003418
Epoch 2/100, Loss: 5.278895378112793
Epoch 3/100, Loss: 4.239290237426758
Epoch 4/100, Loss: 4.222590923309326
Epoch 5/100, Loss: 3.944530725479126
Epoch 6/100, Loss: 3.315995693206787
Epoch 7/100, Loss: 3.04823899269104
Epoch 8/100, Loss: 2.3222804069519043
Epoch 9/100, Loss: 2.04384708404541
Epoch 10/100, Loss: 1.6197398900985718
Epoch 11/100, Loss: 1.3461036682128906
Epoch 12/100, Loss: 1.1274489164352417
Epoch 13/100, Loss: 0.8901744484901428
Epoch 14/100, Loss: 0.644557535648346
Epoch 15/100, Loss: 0.4686383306980133
Epoch 16/100, Loss: 0.35447824001312256
Epoch 17/100, Loss: 0.274071604013443
Epoch 18/100, Loss: 0.21223759651184082
Epoch 19/100, Loss: 0.16042999923229218
Epoch 20/100, Loss: 0.11893784999847412
Epoch 21/100, Loss: 0.08838653564453125
Loss threshold reached. Stopping training.


In [7]:
# Final assertion to ensure loss is below threshold
assert loss.item() < loss_threshold, "Training failed to reduce loss below threshold."
print("Training successful! Loss is below 0.1.")

Training successful! Loss is below 0.1.


In [8]:
prompt = "A quick brown fox"
batch = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False).to(device)

# Generate 18 tokens
for i in range(15):
    # Forward pass to get the logits
    outputs = model(**batch)
    next_token = outputs.logits[0, -1].argmax(-1).reshape(1, 1)

    # Append the next token to input_ids
    batch["input_ids"] = torch.cat([batch["input_ids"], next_token], dim=-1)

    # Update the attention_mask to match the new input_ids length
    new_attention_mask = torch.ones_like(next_token, dtype=batch["attention_mask"].dtype).to(device)
    batch["attention_mask"] = torch.cat([batch["attention_mask"], new_attention_mask], dim=-1)

# Decode the generated sequence
# Skip the virtual tokens (if applicable) by slicing `batch["input_ids"][:, num_prompts:]`
decoded_output = tokenizer.decode(batch["input_ids"][0].cpu().numpy().tolist(), skip_special_tokens=True)
print("\nOutput:", decoded_output)


Output: A quick brown fox did not jump over the lazy dog. Besides, that dog deserved it


### Parameter-efficient finetuning with LoRA (2 points)

When training on more serious tasks, you can use low-rank adapters based on the [LoRA paper](https://arxiv.org/pdf/2106.09685.pdf).

The core idea is to add low-rank adapters __in parallel with existing linear layers,__ like this:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

In the original LoRA paper, the adapters were only added to attention projection matrices. However, [subsequent works](https://arxiv.org/abs/2305.14314) show that it is useful to adapt FFNs as well. But before we do any training, we need to implement the basic LoRA layer.

In [3]:
# re-load the model to remove any previous PEFT tuners
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name, device_map='auto', low_cpu_mem_usage=True, offload_state_dict=True,
    load_in_4bit=True, torch_dtype=torch.float32,  # weights are 4-bit; layernorms and activations are fp32
)
for param in model.parameters():
    param.requires_grad=False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/33 [00:00<?, ?it/s]

In [4]:
class LoRALayer(nn.Module):
    """Wraps a linear layer with LoRA-like adapter. Wraps an existing OPT linear layer"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # pre-trained (frozen) linear layer
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Apply self.module and LoRA adapter, return the sum (self.module outputs + adapter outputs)
        original_output = self.module(input)
        lora_output = input @ self.adapter_A @ self.adapter_B
        
        return original_output + lora_output

In [5]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Apply LoRA to the model

The code below applies LoRA adapters on top of Q/K/V linear layers in Llama attention. You may also choose to modify other layers:
* self_attn.o_proj - attention output projection
* mlp.up_proj, mlp.gate_proj, mlp.down_proj - transformer feedforward layers
* lm_head - output LM head

In [6]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'LlamaDecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 96  # for Llama-7B

In [7]:
batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False)
batch = {k: v.to(device) for k, v in batch.items()}     # А на куду кто двигать будет...

# test a single training step, make sure we get meaningful gradients
with torch.cuda.amp.autocast(dtype=torch.float32):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

C:\Users\Vlad\AppData\Local\Temp\ipykernel_10940\3352942711.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float32):


Grad check successful, well done!


### (example) How to train your model

The example below shows how to train the LoRA adapters on a dummy dataset. You will need to run a _similar_ training task later.

__Note:__ please scroll down for the homework task

In [ ]:
# checking if the model can learn. Change max_steps for proper training
data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 lines
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)
model._hf_peft_config_loaded = True  # silence a warning from HF trainer

trainer = transformers.Trainer(
    model=model, train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, 
        gradient_accumulation_steps=1, # for effectively larger batch size
        warmup_steps=250, 
        max_steps=100, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1,
        output_dir='outputs', 
        report_to=[],       # Тут нельзя писать None, он это не воспринимает!
        save_strategy="no"  # to make it work as of October 2025 (thanks, Vlad!) 
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
# if you see cache warnings, set `model.config.use_cache = False` to silence them. Please re-enable for inference!

trainer.train()

# NOTE: this is just an example! you do not have to wait for this progressbar to finish :)

### Final task: *actually* train the model (5 points)

Your task is to fine-tune the model to _generate python code_. Please use the above examples for inspiration. More specifically,

* __dataset:__ use [codeparrot-clean](https://huggingface.co/datasets/codeparrot/codeparrot-clean) or any other data containing python code. Since you do not need much data for this excercise, it is enough to use just shorter train subset of `codeparrots`
* __preprocessing:__ select python code based on file extentions (.py)  (may skip in case of codeparrot - it is 100% python)
* __short lines:__ please take the first 512 characters of each line
* __adapter type:__ please use LoRA as defined above __plus at least one of:__
   - extra adapter on lm_head
   - extra adapter on MLP components (mlp.*)
   - trainable input embeddings (requires tweaking memory usage)

* __training:__ you do not have to train to convergence. If all goes well, your model should `.generate` code after 500 steps. Please use batch size of at least 4 (4 x 1 x 512 tokens) using `gradient_accumulation_steps=4`.


Note: the peft library also has LoRA implementation. However, we ask that for this assignment you show at least one complete training run with your own LoRA code.

__Alternative assignment:__ Instead of doing python code, feel free to substitute the task with any other dataset, e.g. your favorite artist or podcast, as long as it's ethical. If you choose your own task, please show examples of what your model learned - or did not learn, akin to the code examples below.

Функция для применения LoRA к нескольким частям модели

In [5]:
def apply_lora_to_model(model, rank=8, target_modules=None):
    # Применим LoRA к self attention, ffn + model head
    if target_modules is None:
        target_modules = ['q_proj', 'v_proj', 'mlp.down_proj', 'mlp.up_proj']
    
    lora_count = 0
    for name, module in model.named_modules():
        should_apply = any(target in name for target in target_modules)
        
        if should_apply and isinstance(module, nn.Linear):
            *parent_names, attr_name = name.split('.')
            parent = model
            for pname in parent_names:
                parent = getattr(parent, pname)
            
            lora_layer = LoRALayer(module, rank)
            setattr(parent, attr_name, lora_layer)
            lora_count += 1
            # print(f"Applied LoRA to: {name}")
    
    # lm_head
    if hasattr(model, 'lm_head') and isinstance(model.lm_head, nn.Linear):
        model.lm_head = LoRALayer(model.lm_head, rank)
        lora_count += 1
        print("Applied LoRA to: lm_head")
    
    print(f"\nTotal LoRA layers added: {lora_count}")
    
    # freeze other weights
    for name, param in model.named_parameters():
        if 'adapter_A' in name or 'adapter_B' in name:
            param.requires_grad = True
        else:
            param.requires_grad = False
    
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nTrainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")
    
    return model

model = apply_lora_to_model(
    model, 
    rank=8,
    target_modules=['q_proj', 'v_proj', 'mlp.down_proj', 'mlp.up_proj', 'mlp.gate_proj']
)

Applied LoRA to: lm_head

Total LoRA layers added: 161

Trainable parameters: 16,082,944
Total parameters: 3,516,495,872
Trainable %: 0.4574%


Загрузка и подготовка данных

In [6]:
def load_and_preprocess_data(tokenizer, max_samples=5000, max_length=512):
    dataset = datasets.load_dataset(
        "codeparrot/codeparrot-clean", 
        split=f"train[:{max_samples}]",
        trust_remote_code=True
    )
    
    print(f"Loaded {len(dataset)} samples")
    
    def preprocess_function(examples):
        truncated_content = [content[:max_length] for content in examples['content']]
        tokenized = tokenizer(
            truncated_content,
            truncation=True,
            max_length=max_length,
            padding=False,
            return_tensors=None
        )
        return tokenized
    
    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing dataset"
    )
    
    return tokenized_dataset

train_dataset = load_and_preprocess_data(tokenizer, max_samples=5000, max_length=512)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'codeparrot/codeparrot-clean' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Loaded 5000 samples


Создание сэмплов

In [7]:
def generate_samples(model, tokenizer, prompts, max_new_tokens=100, device='cuda'):
    model.eval()
    samples = {}
    
    with torch.no_grad():
        for prompt in prompts:
            input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
            
            outputs = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
            
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
            samples[prompt] = generated_text
    
    return samples

prompts = ['', 'import', 'from', 'while', 'try', 'if', 'for', 'torch', 'def', 'class', '# This function']
samples_before = generate_samples(model, tokenizer, prompts, max_new_tokens=80, device=device)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Процесс дообучения

In [8]:
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model._hf_peft_config_loaded = True

trainer = transformers.Trainer(
    model=model,
    train_dataset=train_dataset,
    args=transformers.TrainingArguments(
        # Экономим память
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,

        warmup_steps=50,
        max_steps=500,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=25,
        output_dir='outputs',
        report_to=[],
        save_strategy="no",
        save_steps=500,
        optim="adamw_torch",
        weight_decay=0.01,
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Train the model
trainer.train()

Step,Training Loss
25,1.154800
50,1.054800
75,1.150400
100,0.974800
125,1.137400
150,1.071400
175,1.017900
200,1.110600
225,1.044500
250,1.067400


TrainOutput(global_step=500, training_loss=1.0590882759094238, metrics={'train_runtime': 1355.7419, 'train_samples_per_second': 1.475, 'train_steps_per_second': 0.369, 'total_flos': 1.293368426539008e+16, 'train_loss': 1.0590882759094238, 'epoch': 0.4})

In [9]:
samples_after = generate_samples(model, tokenizer, prompts, max_new_tokens=80, device=device)

In [10]:
from IPython.display import HTML, display

table_template = """<table style="border:1px solid black; border-collapse: collapse;" >
  <tr>
    <th style="text-align: center; border:1px solid black; padding: 8px; background-color: #f0f0f0;">PROMPT</th>
    <th style="text-align: center; border:1px solid black; padding: 8px; background-color: #f0f0f0;">BEFORE</th>
    <th style="text-align: center; border:1px solid black; padding: 8px; background-color: #f0f0f0;">AFTER</th>
  </tr>
{}
</table>"""

row_template = '''  <tr>
    <td style="width:20%; border:1px solid black; padding: 8px; vertical-align: top;"><pre style="margin: 0;">{}</pre></td>
    <td style="width:40%; border:1px solid black; padding: 8px; vertical-align: top;"><pre style="margin: 0; white-space: pre-wrap; word-wrap: break-word;">{}</pre></td>
    <td style="width:40%; border:1px solid black; padding: 8px; vertical-align: top;"><pre style="margin: 0; white-space: pre-wrap; word-wrap: break-word;">{}</pre></td>
  </tr>'''

rows = []

for prompt in prompts:
    before_text = samples_before.get(prompt, "N/A")
    after_text = samples_after.get(prompt, "N/A")
    
    before_display = before_text[:200] + ("..." if len(before_text) > 200 else "")
    after_display = after_text[:200] + ("..." if len(after_text) > 200 else "")
    
    before_display = before_display.replace('<', '&lt;').replace('>', '&gt;')
    after_display = after_display.replace('<', '&lt;').replace('>', '&gt;')
    
    prompt_display = repr(prompt) if prompt else "''" 
    
    rows.append(row_template.format(prompt_display, before_display, after_display))

display(HTML(table_template.format('\n'.join(rows))))

PROMPT,BEFORE,AFTER
'',Sidenote: This is the second installment in a series of posts about my recent visit to the National Library of the Kyrgyz Republic in Bishkek. I will be posting more about the library’s history and ac...,#!/usr/bin/env python # -*- coding: utf-8 -*- # # This file is part of Ansible # # Ansible is free software: you can redistribute it and/or modify # it under the terms of the GNU General Public Licens...
'import',import Foundation class TestCase { static var _singleton = nil static var _singletonRetain: TestCase { return _singleton } var expectation: XCTestCaseExpectation? ...,import unittest from django.conf import settings from django.core.management.base import CommandError from django.core.management.base import CommandParser from django.core.management.base import Com...
'from',from . import unittest from . import utils from . import types from . import enums from . import errors from . import messages class TestMessage(messages.Message): pass class TestMessageWitho...,from __future__ import unicode_literals from django.db import models from django.utils import six class TestModel(models.Model): name = models.CharField(max_length=255) address = models.Cha...
'while',while(1){ $x = rand(1000000000000000000000000000000000000000000000000000000000000000000000,"while True: print ""Please enter a name for the directory you want to create."" name = raw_input(""> "") if name == """": print ""Please enter a name for the directory you want to create..."
'try',try again to connect to the device. It looks like you're trying to connect to a device with a different device type. The device type is the manufacturer's name that is used to identify the device in t...,"try: import ujson as json except ImportError: import json from django.core.exceptions import ValidationError from django.core.validators import EMPTY_VALUES, MinLengthValidator, \ MaxLeng..."
'if',"if (isset($this->_blocks['/'])) exit; ?> <h1>Error</h1> <p> Something went wrong in our world. </p> <p> If you are the developer of this site, please contact us at the following address...",if ( ! defined( 'ABSPATH' ) ) { exit; } /** * Adds a default search box to the header. * * @since 3.0.0 * @api public * * @param array $options Options. * @return string HTML. */ function...
'for',"fortran,fortran 77,fortran 90,fortran 95,fortran 2003,fortran 2008,fortran 2010,fortran 2018,fortran 2019,fortran 2020,fortran 2022,fortran","for a time, the world seemed to stand still. ""The world's greatest man has died."" The words of the announcer were heard in all countries. The news was broadcasted through all nations. The whole world ..."
'torch',torchbearer 5:07 AM Great to read your thoughts on the matter. I've always thought of myself as a Christian and I've always had the same thoughts on the matter. I've never really thought of myself as ...,"torch.require('rnn.hmm') class HMM(nn.Module): def __init__(self, dim_h, dim_w, n_states, n_symbols): super(HMM, self).__init__() self.hmm = nn.HMM(dim_h, dim_w,"
'def',"def _(x, y): print(x, y) print(""Hello World"") x = 10 y = 15 def _(x, y): print(x, y) print(""Hello World"") x = 10 y = 15 def _(x, y):","def __init__(self, name, parent=None, **kwargs): """""" :param name: the name of the widget :type name: str :param parent: the parent of the widget :type parent: Widget """""" se..."
'class',class AddToCartForm extends \Google\Protobuf\Internal\Message { /** * @var \Google\Protobuf\FormField */ protected $formField = null; /** * @var \Google\Protobuf\FormField ...,"class FixedWidthLayout def initialize(width, height) @width = width @height = height @height_ratio = height.to_f / @width end def size(width, height) @width * @height_ratio en..."


If you reach this: congratulations! you've completed everything in this practice session.

If you want to dig deeper, try to implement prompt-tuning (for bonus points!).
You can read more about prompt tuning variants in paper [1](https://arxiv.org/abs/2104.08691) or paper [2](https://arxiv.org/abs/2101.00190). Both versions can be implemented by passing trainable prompts as `model.forward(..., past_key_values=your_prompts)`.



### Read more

* How post-training quantization works: https://arxiv.org/abs/2208.07339
* An overview of running large models: https://huggingface.co/docs/accelerate/package_reference/big_modeling
* A general library for different adapter types: https://adapterhub.ml/


### [extra info] Running other models.

This notebook's code can run with other models of similar size, such as [Falcon-7B](https://huggingface.co/tiiuae/falcon-7b), [OPT-6.7B](https://huggingface.co/facebook/opt-6.7b) or [BLOOM-7.1B](https://huggingface.co/bigscience/bloom-7b1). However, they will require minor code tweaks:
1. change the model name in `AutoModelForCausalLM.from_pretrained()` __and__ `AutoTokenizer`
2. In the prompt tuning code, change `model.model.embed_tokens` to refer to the target model's word embeddings. Simply `print(model)` to navigate to them.
3. Change code to add Lora layers - specifically where you what the transformer block components, since those components now have different names.